In [1]:
!pip install -U "bitsandbytes>=0.46.1" "transformers>=4.45.0" accelerate qwen-vl-utils datasets pillow

In [2]:
"""

  VLM BENCHMARK, KITTI Driving Scene Analysis
  Model A: Qwen2.5-VL-7B-Instruct (Alibaba)
  Model B: LLaVA-1.5-7B (Haotian Liu et al.)

"""

import gc, time
import torch
from datasets import load_dataset
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
)

def vram_gb():
    return torch.cuda.memory_allocated() / 1024**3

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("GPU cleared.\n")


# 4-bit configuration

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Dataset


print("Loading KITTI dataset …")
ds        = load_dataset("dpdl-benchmark/kitti", split="validation", streaming=False)
sample    = ds[102]
image     = sample["image"].convert("RGB")
scene_ctx = f"GPS: 49.102, 8.402 | Label: {sample.get('text', 'Intersection')}"
print(f"Image size: {image.size} | {scene_ctx}\n")

TASK = (
    "You are an autonomous vehicle safety system. Analyze this driving scene. "
    "Identify: (1) all visible hazards, (2) road users and their behaviour, "
    "(3) recommended immediate action for the ego vehicle. "
    "Be specific and concise."
)

results = {}

# MODEL A: Qwen2.5-VL-7B-Instruct  (4-bit)

QW_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
print(f"{'='*60}")
print(f"  MODEL A · {QW_ID}  [4-bit NF4]")
print(f"{'='*60}")

torch.cuda.empty_cache()
v0 = vram_gb()

qw_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QW_ID,
    quantization_config=BNB,
    device_map="auto",
)
qw_proc = AutoProcessor.from_pretrained(QW_ID)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": f"Context: {scene_ctx}\n\n{TASK}"},
    ],
}]
text   = qw_proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = qw_proc(text=[text], images=[image], return_tensors="pt").to("cuda")

print("Generating …")
t0 = time.time()
with torch.no_grad():
    out = qw_model.generate(**inputs, max_new_tokens=256, do_sample=False)
elapsed = time.time() - t0

response_qw = qw_proc.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
vram_qw     = vram_gb() - v0
results["Qwen2.5-VL-7B"] = {"runtime": elapsed, "vram": vram_qw, "output": response_qw}

print(f" {elapsed:.1f}s | {vram_qw:.1f} GB VRAM")
print(f"\n OUTPUT:\n{response_qw}\n")

free_gpu(qw_model, qw_proc, inputs, out)

# MODEL B — LLaVA-1.5-7B  (4-bit)

LV_ID = "llava-hf/llava-1.5-7b-hf"
print(f"{'='*60}")
print(f"  MODEL B · {LV_ID}  [4-bit NF4]")
print(f"{'='*60}")

v0 = vram_gb()

lv_model = LlavaForConditionalGeneration.from_pretrained(
    LV_ID,
    quantization_config=BNB,
    device_map="auto",
)
lv_proc = AutoProcessor.from_pretrained(LV_ID)

lv_prompt = f"USER: <image>\nContext: {scene_ctx}\n\n{TASK}\nASSISTANT:"
lv_inputs = lv_proc(text=lv_prompt, images=image, return_tensors="pt").to("cuda")

print("Generating …")
t0 = time.time()
with torch.no_grad():
    out = lv_model.generate(**lv_inputs, max_new_tokens=256, do_sample=False)
elapsed = time.time() - t0

response_lv = lv_proc.decode(out[0][lv_inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
vram_lv     = vram_gb() - v0
results["LLaVA-1.5-7B"] = {"runtime": elapsed, "vram": vram_lv, "output": response_lv}

print(f" {elapsed:.1f}s | {vram_lv:.1f} GB VRAM")
print(f"\n OUTPUT:\n{response_lv}\n")

free_gpu(lv_model, lv_proc, lv_inputs, out)


# COMPARISON TABLE

SEP = "═" * 60
print(f"\n{SEP}")
print("  BENCHMARK RESULTS — KITTI Scene #102")
print(SEP)
print(f"  {'Metric':<25} {'Qwen2.5-VL-7B':>15} {'LLaVA-1.5-7B':>15}")
print(f"  {'─'*25} {'─'*15} {'─'*15}")
print(f"  {'Inference time (s)':<25} {results['Qwen2.5-VL-7B']['runtime']:>15.1f} {results['LLaVA-1.5-7B']['runtime']:>15.1f}")
print(f"  {'VRAM used (GB)':<25} {results['Qwen2.5-VL-7B']['vram']:>15.1f} {results['LLaVA-1.5-7B']['vram']:>15.1f}")
print(f"  {'Precision':<25} {'4-bit NF4':>15} {'4-bit NF4':>15}")
print(f"  {'Params':<25} {'7B':>15} {'7B':>15}")
print(f"  {'Architecture':<25} {'Qwen2.5+ViT':>15} {'Vicuna+CLIP':>15}")
print(f"  {'Release year':<25} {'2024':>15} {'2023':>15}")
print(SEP)

faster = min(results, key=lambda k: results[k]["runtime"])
leaner = min(results, key=lambda k: results[k]["vram"])
print(f"\n  Faster: {faster}")
print(f"  Leaner: {leaner}")

print(f"\n{SEP}")
print("SIDE-BY-SIDE OUTPUT COMPARISON")
print(SEP)
for name, r in results.items():
    print(f"\n  ── {name} ──")
    print(r["output"])
print(f"\n{SEP}\n")

Loading KITTI dataset …


README.md:   0%|          | 0.00/966 [00:00<?, ?B/s]

data/train-00000-of-00010.parquet:   0%|          | 0.00/482M [00:00<?, ?B/s]

data/train-00001-of-00010.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

data/train-00002-of-00010.parquet:   0%|          | 0.00/482M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Generating …
 10.9s | 1.5 GB VRAM

 OUTPUT:
1. All visible hazards: There are multiple cars on the street, some of which are parked, and a few pedestrians.
2. Road users and their behavior: The cars are driving down the street, and some pedestrians are walking on the sidewalk.
3. Recommended immediate action for the ego vehicle: The ego vehicle should maintain a safe distance from the parked cars and be cautious while navigating the street, especially around pedestrians.

GPU cleared.


════════════════════════════════════════════════════════════
  BENCHMARK RESULTS — KITTI Scene #102
════════════════════════════════════════════════════════════
  Metric                      Qwen2.5-VL-7B    LLaVA-1.5-7B
  ───────────────────────── ─────────────── ───────────────
  Inference time (s)                   28.3            10.9
  VRAM used (GB)                        1.5             1.5
  Precision                       4-bit NF4       4-bit NF4
  Params                                 7B    